# IRWA 2026 Part 2: Indexing and Evaluation

Structure:

0. Setup and data loading
1. **PART 1: Indexing**
   - 1.1 Build inverted index (conjunctive AND queries)
   - 1.2 Propose test queries
   - 1.3 Rank your results: TF-IDF
2. **PART 2: Evaluation**
   - 2.1 Evaluation metrics
   - 2.2 Validation queries (`validation_labels.csv`)
   - 2.3 Expert judges: our test queries
     - 2.3.a Relevance labels
     - 2.3.b Comments on the evaluation metrics
     - 2.3.c Analysis of the search system: problems and solutions

The functions live in `myapp/search/algorithms.py` (indexing, search, TF-IDF) and
`myapp/search/evaluation.py` (metrics), numbered with the same points (1.1, 1.3, 2.1).

## 0. Setup and data loading

In [1]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("../..").resolve()
sys.path.append(str(ROOT))

DATA_PATH = ROOT / "data" / "fashion_products_dataset.json"
CORPUS_PATH = ROOT / "data" / "processed_corpus.pkl" 
LABELS_PATH = ROOT / "data" / "validation_labels.csv"
OUR_LABELS_PATH = Path("our_queries_labels.csv") # our own judgements (2.3.a)

from myapp.search.preprocessing import preprocess_text, load_raw_corpus, build_processed_corpus
from myapp.search.algorithms import (
    create_index, conjunctive_search, # 1.1
    compute_idf, compute_doc_norms, rank_tfidf, search_in_corpus) # 1.3
from myapp.search.evaluation import (
    ranked_labels, precision_at_k, recall_at_k, avg_precision_at_k, f1_at_k, # 2.1
    mean_average_precision, reciprocal_rank, mean_reciprocal_rank, ndcg_at_k, evaluate_query)

pd.set_option("display.max_colwidth", 60)

In [2]:
# Processed corpus from Part 1 (rebuilt from the raw JSON if the file is missing or unreadable)
try:
    corpus = pd.read_pickle(CORPUS_PATH)
except Exception:
    corpus = build_processed_corpus(load_raw_corpus(DATA_PATH))
    corpus.to_pickle(CORPUS_PATH)

corpus_by_pid = corpus.set_index("pid")
print(corpus.shape)
corpus[["pid", "title", "all_tokens"]].head(3)

(28080, 28)


,pid,title,all_tokens
0,TKPFCZ9EA7H5FYZH,Solid Women Multicolor Track Pants,"[solid, women, multicolor, track, pant, yorker, trackpan..."
1,TKPFCZ9EJZV2UVRZ,Solid Men Blue Track Pants,"[solid, men, blue, track, pant, yorker, trackpant, made,..."
2,TKPFCZ9EHFCY5Z4Y,Solid Men Multicolor Track Pants,"[solid, men, multicolor, track, pant, yorker, trackpant,..."


## PART 1: Indexing

### 1.1 Build inverted index

`create_index()` builds `{term: {pid: tf}}` from the merged field `all_tokens` (Part 1, 1.3):
for each term, the products that contain it and how many times. The counts (tf) are needed for TF-IDF in 1.3.

`conjunctive_search()` implements the **AND** queries: a product is returned only if it contains
**all** the query terms (intersection of the terms' lists).

In [3]:
start = time.time()
index = create_index(corpus, field="all_tokens")
print(f"Index built in {time.time() - start:.1f} s")
print("Vocabulary size (terms):", len(index))
print("Products indexed:", len({pid for postings in index.values() for pid in postings}))

Index built in 0.8 s
Vocabulary size (terms): 5790
Products indexed: 28080


In [4]:
# Example: the posting list of one term
term = "sweatshirt"
print(f"'{term}' appears in {len(index[term])} products, e.g.:")
list(index[term].items())[:5]

'sweatshirt' appears in 1417 products, e.g.:


[('SWSFYD33P2XCBWFG', 1),
 ('SWSFYD3KMGTBZEJZ', 1),
 ('SWSFYD38Y2MAT9FQ', 1),
 ('SWSFVEV2YFEJKUCK', 2),
 ('SWSFVEV2BWZJDJ46', 1)]

In [5]:
# Conjunctive (AND) search on the two validation queries
for q in ["women full sleeve sweatshirt cotton", "men slim jeans blue"]:
    terms = preprocess_text(q)
    print(q, "->", terms)
    for t in terms:
        print(f"   {t:12} in {len(index.get(t, {})):6} products")
    print(f"   AND (all terms) -> {len(conjunctive_search(terms, index))} products\n")

women full sleeve sweatshirt cotton -> ['women', 'full', 'sleev', 'sweatshirt', 'cotton']
   women        in  13699 products
   full         in   7150 products
   sleev        in  17904 products
   sweatshirt   in   1417 products
   cotton       in  19584 products
   AND (all terms) -> 467 products

men slim jeans blue -> ['men', 'slim', 'jean', 'blue']
   men          in  14123 products
   slim         in   5646 products
   jean         in   2430 products
   blue         in   7154 products
   AND (all terms) -> 219 products



### 1.2 Propose test queries

The queries are built from **popular terms** (highest document frequency): a gender word, an
attribute (fabric, colour, pattern, occasion) and a product type, so they look like real shop searches.
Each query covers a different sub-category and returns enough results under the AND rule.

In [6]:
# Most popular terms: number of products that contain each term (document frequency)
df = pd.Series({term: len(postings) for term, postings in index.items()}).sort_values(ascending=False)
df.head(30).to_frame("products")

,products
accessori,27189
cloth,27121
wear,20925
cotton,19584
western,19095
sleev,17904
topwear,15310
1,15106
neck,14569
men,14123


In [7]:
#Each query target different product categories, so we can use them to test the search engine's ability to retrieve relevant products across different categories.
TEST_QUERIES = {                            #targeted product sub-categories
    3: "women cotton printed kurta",      # Kurtas, Ethnic Sets and Bottoms
    4: "men black polo tshirt",           # Topwear
    5: "women winter hooded jacket",      # Winter Wear
    6: "men running sports shoes",        # Men's Footwear
    7: "men formal trousers",             # Bottomwear
}

pd.DataFrame([{"query_id": qid, "query": q, "terms": preprocess_text(q),
               "AND results": len(conjunctive_search(preprocess_text(q), index))}
              for qid, q in TEST_QUERIES.items()])

,query_id,query,terms,AND results
0,3,women cotton printed kurta,"[women, cotton, print, kurta]",41
1,4,men black polo tshirt,"[men, black, polo, tshirt]",222
2,5,women winter hooded jacket,"[women, winter, hood, jacket]",72
3,6,men running sports shoes,"[men, run, sport, shoe]",74
4,7,men formal trousers,"[men, formal, trouser]",237


### 1.3 Rank your results: TF-IDF

Each term in a product gets a weight, and products are ranked by the **cosine similarity**
between the query vector and the product vector:

$$
w_{t,d} = (1 + \log tf_{t,d}) \cdot \log\frac{N}{df_t} \qquad
score(q, d) = \frac{\sum_{t \in q} w_{t,q} \, w_{t,d}}{\lVert q \rVert \, \lVert d \rVert}
$$

* `tf`: times the term appears in the product (log-scaled, so 10 repetitions do not count 10×).
* `idf = log(N / df)`: rare terms weigh more than terms found in almost every product.
* Dividing by the vector lengths stops long descriptions from winning just because they are long.

In [8]:
start = time.time()
N = len(corpus)
idf = compute_idf(index, N)
doc_norms = compute_doc_norms(index, idf)
print(f"IDF and document norms computed in {time.time() - start:.1f} s")

pd.Series({t: idf[t] for q in ["women full sleeve sweatshirt cotton", "men slim jeans blue"]
           for t in preprocess_text(q)}).round(3).to_frame("idf")

IDF and document norms computed in 1.1 s


,idf
women,0.718
full,1.368
sleev,0.450
sweatshirt,2.987
cotton,0.360
men,0.687
slim,1.604
jean,2.447
blue,1.367


In [9]:
SHOW = ["pid", "title", "brand", "selling_price", "average_rating", "out_of_stock"]

def show_results(query, k=10):
    """Top-k results of a query: rank, score and some of the output fields."""
    ranking = search_in_corpus(query, index, idf, doc_norms, top_k=k)
    table = corpus_by_pid.loc[[pid for pid, _ in ranking]].reset_index()[SHOW]
    table.insert(0, "score", [round(score, 3) for _, score in ranking])
    table.index = range(1, len(table) + 1)
    return table

ALL_QUERIES = {1: "women full sleeve sweatshirt cotton", 2: "men slim jeans blue", **TEST_QUERIES}
for qid, q in ALL_QUERIES.items():
    print(f"Query {qid}: {q}")
    display(show_results(q))

Query 1: women full sleeve sweatshirt cotton


,score,pid,title,brand,selling_price,average_rating,out_of_stock
1,0.747,SWSFZVTTQCB4SJ7F,Full Sleeve Solid Women Sweatshirt,Lev,2239.0,NaN,True
2,0.729,SWSFZVTNGM4HG8BC,Full Sleeve Printed Women Sweatshirt,Lev,1042.0,4.2,False
3,0.702,SWSEXAHDUWHFZDHC,Full Sleeve Printed Women Sweatshirt,G,3594.0,3.7,False
4,0.697,SWSEXAHDQQCKCAEA,Full Sleeve Printed Women Sweatshirt,G,3385.0,4.0,False
5,0.685,SWSFQGS456JAZCQB,Full Sleeve Printed Women Sweatshirt,Lev,2079.0,NaN,True
6,0.678,SWSFV5JNFW5HZQWY,Full Sleeve Printed Women Sweatshirt,ECKO Unl,769.0,2.5,False
7,0.660,SWSFYRKYAHH4HHSM,Full Sleeve Printed Women Sweatshirt,Steenb,849.0,3.7,False
8,0.652,SWSFY37UKWPG4AZ2,Full Sleeve Solid Women Sweatshirt,ECKO Unl,725.0,2.5,False
9,0.652,SWSFZ2H4KMYXZXX7,Full Sleeve Solid Women Sweatshirt,ECKO Unl,747.0,2.5,False
10,0.652,SWSFYTYMNTBNARUN,Full Sleeve Solid Women Sweatshirt,ARBO,759.0,4.2,False


Query 2: men slim jeans blue


,score,pid,title,brand,selling_price,average_rating,out_of_stock
1,0.592,JEAFSGSYEAFCYHEE,Slim Men Blue Jeans,Lev,2033.0,NaN,True
2,0.592,JEAFTGSGTYKZGAEZ,Slim Men Blue Jeans,Lev,3679.0,NaN,True
3,0.583,JEAFSKYHBSMZMH8F,Slim Men Blue Jeans,ECKO Unl,974.0,3.2,False
4,0.583,JEAFSKYHZHSZZC9S,Slim Men Blue Jeans,ECKO Unl,899.0,2.5,False
5,0.578,JEAFEKQZ4C2Z6GCX,Slim Men Blue Jeans,True Bl,1199.0,3.6,False
6,0.578,JEAFEKQZQPYHHVGS,Slim Men Blue Jeans,True Bl,1199.0,3.6,False
7,0.578,JEAFEKQZWYPVGYCJ,Slim Men Blue Jeans,True Bl,1199.0,4.3,False
8,0.564,JEAFESND4QWQUBZD,Slim Men Blue Jeans,True Bl,1319.0,5.0,False
9,0.556,JEAFSKYHSYMTKEJD,Slim Men Blue Jeans,ECKO Unl,899.0,3.1,False
10,0.556,JEAFSKYHUXWYKBZK,Slim Men Blue Jeans,ECKO Unl,974.0,4.1,False


Query 3: women cotton printed kurta


,score,pid,title,brand,selling_price,average_rating,out_of_stock
1,0.433,FABFWN82HVT5KAH2,Cotton Printed Kurta Fabric (Unstitched),Fa,699.0,1.7,False
2,0.427,ETHFT9HQDDYTZZT5,Women Kurta and Pyjama Set Cotton Blend,Jagdish Garmen,1035.0,3.0,False
3,0.427,ETHFT9HQNKWEF2EH,Women Kurta and Pyjama Set Cotton Blend,Jagdish Garmen,1035.0,3.0,False
4,0.427,ETHFT9HQRGEXHETK,Women Kurta and Pyjama Set Cotton Blend,Jagdish Garmen,1035.0,3.0,False
5,0.425,KTAFZCK5RGNECHHH,Women Printed Cotton Linen Blend Straight Kurta (Blue),Mah,519.0,4.1,False
6,0.419,KTAFZCH5BCNAFZJ6,Women Printed Cotton Linen Blend Straight Kurta (White),Mah,519.0,4.1,False
7,0.401,KTAFZCH5954HZSWJ,Women Printed Cotton Linen Blend Straight Kurta (White),Mah,519.0,4.1,False
8,0.400,KTAF7YABEAXUHWGF,Women Printed Cotton Silk Straight Kurta (Beige),Purple Sta,719.0,3.3,False
9,0.392,ETHFY7FBXY6ZDTKN,Women Kurta and Pyjama Set Pure Cotton,Jagdish Garmen,724.0,NaN,False
10,0.392,ETHFY7FBZZYWK2ZH,Women Kurta and Pyjama Set Pure Cotton,Jagdish Garmen,724.0,NaN,False


... (4 outputs left out)

Query 6: men running sports shoes


,score,pid,title,brand,selling_price,average_rating,out_of_stock
1,0.690,SHOESS9ZEZ6TEQUJ,Running Shoes For Men (Multicolor),Vector,1440.0,4.3,False
2,0.690,SHOESS9ZGMH7CURM,Running Shoes For Men (Multicolor),Vector,1440.0,3.7,False
3,0.676,SHOEP5GUQYYFKYSN,Running Shoes For Women (White),Vector,990.0,4.1,False
4,0.595,SHOEWQKVSGFGXPQF,Running Shoes For Men (Blue),Vector,1520.0,3.3,False
5,0.585,SHOFTZFPNCU4VWHK,Running Shoes For Men (Black),Bone,379.0,4.1,False
6,0.579,SHOFTZFQP8W9G86F,Running Shoes For Women (Black),Bone,341.0,4.1,False
7,0.560,SHOFN3EJHXHGSGUC,RS-7250 Running Shoes For Men (Black),Vector,990.0,3.9,False
8,0.557,SHOEKNRJGEYJCDQK,"SPRINT Running Shoes For Men (Blue, Grey)",Vector,490.0,3.7,False
9,0.556,SHOFTZFQQYKC8HY6,Running Shoes For Men (Yellow),Bone,341.0,4.1,False
10,0.556,SHOEKNRJ34UCMAPA,"SPRINT Running Shoes For Men (Black, Grey)",Vector,490.0,3.7,False


Query 7: men formal trousers


,score,pid,title,brand,selling_price,average_rating,out_of_stock
1,0.739,TROFZD64MABMGW5K,Slim Fit Men White Pure Cotton Trousers,K,735.0,2.5,False
2,0.694,TROFM47EV9ZNVDJF,Slim Fit Men Blue Cotton Blend Trousers,True Bl,999.0,3.6,False
3,0.694,TROFM47GWDMQZKR9,Slim Fit Men Blue Cotton Blend Trousers,True Bl,999.0,3.6,False
4,0.689,TROFZD7DT2FCCJ4H,Slim Fit Men Dark Blue Pure Cotton Trousers,K,735.0,2.5,False
5,0.680,TROFM47FSDAMFGQU,Slim Fit Men Blue Cotton Blend Trousers,True Bl,1039.0,3.6,False
6,0.642,TROFM47FEMXGWP4P,Slim Fit Men Blue Cotton Blend Trousers,True Bl,799.0,3.6,False
7,0.633,TROFZ3P9BZZZYCBV,Regular Fit Men Black Cotton Blend Trousers,Crocks Cl,999.0,3.0,False
8,0.614,TROFNNPG5YPUBPZU,Slim Fit Men Green Cotton Blend Trousers,True Bl,799.0,4.4,False
9,0.614,TROFNMG4G9DG6QEF,Slim Fit Men Black Polyester Trousers,True Bl,1039.0,2.6,False
10,0.607,TROFM47DXRTJWXYC,Slim Fit Men Beige Cotton Blend Trousers,True Bl,999.0,3.6,False


## PART 2: Evaluation

### 2.1 Evaluation metrics

All metrics (in `evaluation.py`) receive `rels`, the relevance labels of the results in ranking
order (e.g. `[1, 0, 1, ...]`), and `total_relevant`, the number of relevant products for the query.

| Metric | Function | Measures |
|---|---|---|
| Precision@K | `precision_at_k` | relevant results among the first K |
| Recall@K | `recall_at_k` | share of all relevant products found in the first K |
| Average Precision@K | `avg_precision_at_k` | precision at each relevant position, divided by min(total relevant, K) |
| F1-Score@K | `f1_at_k` | harmonic mean of P@K and R@K |
| MAP | `mean_average_precision` | mean of AP@K over the queries |
| MRR | `mean_reciprocal_rank` | mean of 1 / position of the first relevant result |
| NDCG@K | `ndcg_at_k` | DCG (relevant results discounted by log2 of their position) / DCG of the perfect ranking |

In [10]:
# Check on a small example: 10 results, 5 relevant products in total
rels, total = [1, 0, 1, 1, 0, 0, 1, 0, 0, 0], 5
print("rels =", rels, "| total relevant =", total)
for name, value in evaluate_query(rels, 10, total).items():
    print(f"{name:8} {value:.3f}")
print("MAP (with a 2nd query [0, 1, 0]) :", round(mean_average_precision([(rels, total), ([0, 1, 0], 1)], 10), 3))
print("MRR (with a 2nd query [0, 1, 0]) :", round(mean_reciprocal_rank([rels, [0, 1, 0]], 10), 3))

rels = [1, 0, 1, 1, 0, 0, 1, 0, 0, 0] | total relevant = 5
P@10     0.400
R@10     0.800
AP@10    0.598
F1@10    0.533
RR@10    1.000
NDCG@10  0.768
MAP (with a 2nd query [0, 1, 0]) : 0.549
MRR (with a 2nd query [0, 1, 0]) : 0.75


### 2.2 Validation queries (`validation_labels.csv`)

Setup: K = 10. The ranking is our AND + TF-IDF result list, keeping only the labelled products
(unlabelled products cannot be judged). Labelled relevant products that the AND search does not
return count as missed (they lower recall).

In [11]:
K = 10

def evaluate_queries(queries, labels_df, k=K):
    """Run each query, compare with the labels and compute every metric (rounded to 3 decimals)."""
    rows, per_query = {}, []
    for qid, q in queries.items():
        truth = labels_df[labels_df["query_id"] == qid]
        labels = dict(zip(truth["pid"], truth["labels"]))
        total_relevant = int(truth["labels"].sum())
        ranking = [pid for pid, _ in search_in_corpus(q, index, idf, doc_norms)]
        rels = ranked_labels(ranking, labels)
        rows[f"Query {qid}"] = evaluate_query(rels, k, total_relevant)
        per_query.append((rels, total_relevant))
    table = pd.DataFrame(rows).T
    summary = pd.DataFrame({"MAP": [mean_average_precision(per_query, k)],
                            "MRR": [mean_reciprocal_rank([r for r, _ in per_query], k)]}, index=["All queries"])
    return table.round(3), summary.round(3)

validation = pd.read_csv(LABELS_PATH, dtype={"pid": str})
VALIDATION_QUERIES = {1: "women full sleeve sweatshirt cotton", 2: "men slim jeans blue"}
val_table, val_summary = evaluate_queries(VALIDATION_QUERIES, validation)
display(val_table)
display(val_summary)

,P@10,R@10,AP@10,F1@10,RR@10,NDCG@10
Query 1,1.0,0.769,1.000,0.87,1.0,1.000
Query 2,0.6,0.600,0.569,0.60,1.0,0.716


,MAP,MRR
All queries,0.785,1.0


### 2.3 Expert judges: our test queries

#### 2.3.a Relevance labels

We judge the **top 20 results** of each test query: 1 = relevant, 0 = not relevant.

The next cell writes `our_queries_labels_template.csv` (in this folder) with the top 20 products per query. So we opened it in Excel, fill the `labels` column with 1 or 0, and saved it as **`our_queries_labels.csv`** (CSV format). Then, we've run the next cell to evaluate it.

In [12]:
TEMPLATE_PATH = Path("our_queries_labels_template.csv")
rows = []
for qid, q in TEST_QUERIES.items():
    for pid, _ in search_in_corpus(q, index, idf, doc_norms, top_k=20):
        product = corpus_by_pid.loc[pid]
        rows.append({"title": product["title"], "pid": pid, "query_id": qid, "labels": "",
                     "query": q, "brand": product["brand"], "sub_category": product["sub_category"],
                     "details": product["details_text"]})
pd.DataFrame(rows).to_csv(TEMPLATE_PATH, index=False)
print(f"Template with {len(rows)} products written to project_progress/part_2/{TEMPLATE_PATH}")

Template with 100 products written to project_progress/part_2/our_queries_labels_template.csv


In [13]:
if OUR_LABELS_PATH.exists():
    ours = pd.read_csv(OUR_LABELS_PATH, dtype={"pid": str})
    ours["labels"] = ours["labels"].astype(int)
    print("Relevant products per query:", ours.groupby("query_id")["labels"].sum().to_dict())
    our_table, our_summary = evaluate_queries(TEST_QUERIES, ours)
    display(our_table)
    display(our_summary)
else:
    print(f"{OUR_LABELS_PATH} not found yet: fill the template (2.3.a) and save it with this name.")

Relevant products per query: {3: 18, 4: 20, 5: 14, 6: 12, 7: 20}


,P@10,R@10,AP@10,F1@10,RR@10,NDCG@10
Query 3,0.9,0.500,0.707,0.643,0.5,0.780
Query 4,1.0,0.500,1.000,0.667,1.0,1.000
Query 5,0.7,0.500,0.700,0.583,1.0,0.801
Query 6,0.8,0.667,0.659,0.727,1.0,0.812
Query 7,1.0,0.500,1.000,0.667,1.0,1.000


,MAP,MRR
All queries,0.813,0.9


#### 2.3.b Comments on the evaluation metrics

In the report.

#### 2.3.c Analysis of the search system: problems and solutions

The cells below give evidence for the analysis (retrieval accuracy, ranking quality, field types,
query formulation, indexing).

In [14]:
# 1. AND is strict: labelled relevant products that the search does NOT return, and the terms they miss
for qid, q in VALIDATION_QUERIES.items():
    terms = preprocess_text(q)
    retrieved = conjunctive_search(terms, index)
    truth = validation[(validation["query_id"] == qid) & (validation["labels"] == 1)]
    missed = truth[~truth["pid"].isin(retrieved)]
    print(f"Query {qid}: {len(missed)} of {len(truth)} relevant products not retrieved")
    for pid in missed["pid"]:
        tokens = set(corpus_by_pid.loc[pid, "all_tokens"])
        print(f"   {corpus_by_pid.loc[pid, 'title'][:45]:45} missing terms: {[t for t in terms if t not in tokens]}")

Query 1: 1 of 13 relevant products not retrieved
   Full Sleeve Solid Women Sweatshirt            missing terms: ['cotton']
Query 2: 4 of 10 relevant products not retrieved
   Slim Women Blue Jeans                         missing terms: ['men']
   Super Skinny Women Blue Jeans                 missing terms: ['men']
   Tapered Fit Men Blue Jeans                    missing terms: ['slim']
   Super Skinny Women Blue Jeans                 missing terms: ['men']


In [15]:
# 2. Number of results: AND vs OR (any term)
pd.DataFrame([{"query": q,
               "AND results": len(conjunctive_search(preprocess_text(q), index)),
               "OR results": len(set().union(*[index.get(t, {}).keys() for t in preprocess_text(q)]))}
              for q in ALL_QUERIES.values()])

,query,AND results,OR results
0,women full sleeve sweatshirt cotton,467,25503
1,men slim jeans blue,219,19959
2,women cotton printed kurta,41,24517
3,men black polo tshirt,222,21875
4,women winter hooded jacket,72,15435
5,men running sports shoes,74,15033
6,men formal trousers,237,15791


In [16]:
# 3. Ranking quality: duplicates, out-of-stock products and ratings in the top 10
rows = []
for qid, q in ALL_QUERIES.items():
    top = show_results(q, k=10)
    rows.append({"query": q, "distinct titles in top 10": top["title"].nunique(),
                 "out of stock in top 10": int(top["out_of_stock"].sum()),
                 "avg rating top 10": round(top["average_rating"].mean(), 2),
                 "avg rating all results": round(corpus_by_pid.loc[[p for p, _ in search_in_corpus(q, index, idf, doc_norms)], "average_rating"].mean(), 2)})
pd.DataFrame(rows)

,query,distinct titles in top 10,out of stock in top 10,avg rating top 10,avg rating all results
0,women full sleeve sweatshirt cotton,2,2,3.41,3.80
1,men slim jeans blue,1,2,3.68,3.27
2,women cotton printed kurta,6,0,3.29,3.79
3,men black polo tshirt,2,1,3.82,3.67
4,women winter hooded jacket,3,1,3.97,3.70
5,men running sports shoes,9,0,3.90,3.84
6,men formal trousers,7,0,3.30,3.44


In [17]:
# 4. Field types: labelled products returned by the search but NOT relevant (query 2)
truth = validation[validation["query_id"] == 2].set_index("pid")["labels"]
ranking = [pid for pid, _ in search_in_corpus("men slim jeans blue", index, idf, doc_norms)]
wrong = [pid for pid in ranking if truth.get(pid) == 0]
corpus_by_pid.loc[wrong, ["title", "brand", "details_text", "description"]]

,title,brand,details_text,description
pid,,,,
JEAFUZXTN7W8ZSEN,Tapered Fit Men Blue Jeans,ECKO Unl,Cotton Blend Solid Men Blue Jeans Western Wear Mid Rise,ECKO Unltd SLIM TAPERED Cotton Stretch INDIGO Jeans
JEAFHEZH9KVGTJJS,Slim Men Blue Jeans,U.S. POLO ASS,Pure Cotton Solid Men Blue Button Western Wear Mid Rise,BrandonLooping washBlue
